# ROLAND (You, Du, Leskovec, KDD 2022) — minimal PyTorch/PyG implementation
Moving-Avg / MLP / GRU embedding updates, live-update eval (Alg.2), Reptile meta-training (Alg.3).

In [1]:
!pip install -q torch_geometric


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.3 MB/s eta 0:00:00


In [2]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
from torch_geometric.nn import SAGEConv
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [3]:
# ---- synthetic dynamic graph (swap with real snapshots) ----
def make_snapshots(T=20, N=200, E=400, F_dim=16):
    return [(torch.randn(N, F_dim, device=device),
             torch.randint(0, N, (2, E), device=device)) for _ in range(T)]


In [4]:
# ---- embedding update modules (Eq. 2-3) ----
class MovingAvg(nn.Module):
    def forward(self, h_prev, h_new, kappa):
        return kappa * h_prev + (1 - kappa) * h_new

class MLPUpdate(nn.Module):
    def __init__(self, d):
        super().__init__(); self.mlp = nn.Sequential(nn.Linear(2*d, d), nn.ReLU(), nn.Linear(d, d))
    def forward(self, h_prev, h_new, kappa=None):
        return self.mlp(torch.cat([h_prev, h_new], -1))

class GRUUpdate(nn.Module):
    def __init__(self, d):
        super().__init__(); self.gru = nn.GRUCell(d, d)
    def forward(self, h_prev, h_new, kappa=None):
        return self.gru(h_new, h_prev)

UPDATES = {'moving_average': MovingAvg, 'mlp': MLPUpdate, 'gru': GRUUpdate}


In [5]:
# ---- ROLAND model: static GNN (SAGEConv + skip + BN) repurposed for dynamic graphs ----
class ROLAND(nn.Module):
    def __init__(self, in_dim, hidden=64, layers=2, update='gru'):
        super().__init__()
        self.L = layers; self.update_name = update
        self.pre = nn.Linear(in_dim, hidden)
        self.convs = nn.ModuleList([SAGEConv(hidden, hidden) for _ in range(layers)])
        self.norms = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(layers)])
        self.upd = nn.ModuleList([UPDATES[update](hidden) for _ in range(layers)])
        self.link_mlp = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x, edge_index, H_prev, kappa=1.0):
        h = self.pre(x); H_new = []
        for l in range(self.L):
            h_tilde = F.relu(self.norms[l](self.convs[l](h, edge_index) + h))  # skip-connection
            if H_prev[l] is None:
                h = h_tilde
            else:
                args = (H_prev[l], h_tilde, kappa) if self.update_name == 'moving_average' else (H_prev[l], h_tilde)
                h = self.upd[l](*args)
            H_new.append(h)
        return h, H_new

    def predict(self, h, edge_index):
        s, d = edge_index
        return self.link_mlp(torch.cat([h[s], h[d]], -1)).squeeze(-1)


In [6]:
# ---- MRR metric ----
def mrr(model, h, pos_edges, num_nodes, num_neg=100):
    ranks = []
    for u, v in pos_edges.t().tolist():
        neg = torch.randint(0, num_nodes, (num_neg,), device=device)
        cand = torch.cat([torch.tensor([v], device=device), neg])
        src = torch.full_like(cand, u)
        scores = model.predict(h, torch.stack([src, cand]))
        ranks.append(1.0 / ((scores > scores[0]).sum().item() + 1))
    return float(np.mean(ranks))


In [7]:
# ---- live-update training/eval (Algorithm 2): finetune on G_t, predict G_{t+1} ----
def live_update(model, snapshots, layers, lr=0.01, epochs=20):
    H = [None]*layers; opt = torch.optim.Adam(model.parameters(), lr=lr)
    cum_edges, mrrs = 0, []
    for t in range(len(snapshots) - 1):
        x, ei = snapshots[t]
        kappa = cum_edges / (cum_edges + ei.size(1) + 1e-9); cum_edges += ei.size(1)
        for _ in range(epochs):
            opt.zero_grad()
            h, _ = model(x, ei, H, kappa)
            pos = model.predict(h, ei)
            neg_ei = torch.randint(0, x.size(0), ei.shape, device=device)
            neg = model.predict(h, neg_ei)
            loss = F.binary_cross_entropy_with_logits(pos, torch.ones_like(pos)) + \
                   F.binary_cross_entropy_with_logits(neg, torch.zeros_like(neg))
            loss.backward(); opt.step()
        with torch.no_grad():
            h, H = model(x, ei, H, kappa); H = [hh.detach() for hh in H]
        x_n, ei_n = snapshots[t+1]
        with torch.no_grad():
            h_n, _ = model(x_n, ei_n, H, kappa)
            m = mrr(model, h_n, ei_n, x.size(0)); mrrs.append(m)
        print(f't={t} MRR={m:.4f}')
    return mrrs


In [8]:
# ---- Reptile meta-training (Algorithm 3) ----
def reptile_train(meta_model, snapshots, in_dim, hidden, layers, alpha=0.5, inner_lr=0.01, inner_steps=20):
    meta_state = {k: v.clone() for k, v in meta_model.state_dict().items()}
    H, cum_edges = [None]*layers, 0
    for t in range(len(snapshots) - 1):
        model = ROLAND(in_dim, hidden, layers, meta_model.update_name).to(device)
        model.load_state_dict(meta_state)
        opt = torch.optim.Adam(model.parameters(), lr=inner_lr)
        x, ei = snapshots[t]
        kappa = cum_edges / (cum_edges + ei.size(1) + 1e-9); cum_edges += ei.size(1)
        for _ in range(inner_steps):
            opt.zero_grad()
            h, _ = model(x, ei, H, kappa)
            pos = model.predict(h, ei)
            neg_ei = torch.randint(0, x.size(0), ei.shape, device=device)
            neg = model.predict(h, neg_ei)
            loss = F.binary_cross_entropy_with_logits(pos, torch.ones_like(pos)) + \
                   F.binary_cross_entropy_with_logits(neg, torch.zeros_like(neg))
            loss.backward(); opt.step()
        with torch.no_grad():
            h, H = model(x, ei, H, kappa); H = [hh.detach() for hh in H]
        new_state = model.state_dict()
        meta_state = {k: (1-alpha)*meta_state[k] + alpha*new_state[k] for k in meta_state}
    meta_model.load_state_dict(meta_state)
    return meta_model


In [9]:
# ---- run ----
snapshots = make_snapshots()
model = ROLAND(in_dim=16, hidden=64, layers=2, update='gru').to(device)
mrrs = live_update(model, snapshots, layers=2)
print('Average MRR:', np.mean(mrrs))


t=0 MRR=0.0561
t=1 MRR=0.0580
t=2 MRR=0.0925
t=3 MRR=0.0985
t=4 MRR=0.1007
t=5 MRR=0.1232
t=6 MRR=0.1368
t=7 MRR=0.1428
t=8 MRR=0.1252
t=9 MRR=0.1294
t=10 MRR=0.1385
t=11 MRR=0.1337
t=12 MRR=0.1467
t=13 MRR=0.1260
t=14 MRR=0.1268
t=15 MRR=0.1503
t=16 MRR=0.1371
t=17 MRR=0.1311
t=18 MRR=0.1345
Average MRR: 0.12042791954075682
